# 📊 Разведочный анализ данных (EDA) — Страховка «Защищённый смартфон» (Сбер)

Данный ноутбук содержит полный исследовательский анализ обучающей выборки (`train.csv`) и тестовой выборки (`public_test.csv`) для задачи предсказания покупки страхового продукта.


In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

train = pd.read_csv('train.csv')
test = pd.read_csv('public_test.csv')

print(f"Размер train: {train.shape}")
print(f"Размер test: {test.shape}")
train.head()


## 1. Анализ целевой переменной `accepted`


In [ ]:
target_counts = train['accepted'].value_counts()
target_pct = train['accepted'].value_counts(normalize=True) * 100

print("Распределение целевой переменной:")
for val in [0, 1]:
    print(f"Класс {val}: {target_counts[val]} ({target_pct[val]:.2f}%)")


## 2. Анализ пропущенных значений и доменная логика
- `loan_amount`: отсутствует ровно у тех клиентов, у которых `has_credit == 0` (сумма кредита = 0 руб.).
- `insurance_claims`: отсутствует ровно у тех клиентов, у которых `previous_insurance == 0` (нет страховых случаев, так как не было страховок).
- `credit_score`: отсутствует у части клиентов без кредитной истории.
- `mobile_app_usage`: отсутствует у клиентов без мобильного приложения (0 сессий/нед).


In [ ]:
missing_train = train.isnull().sum()
missing_train = missing_train[missing_train > 0]
print("Пропуски в обучающей выборке:")
print(missing_train)


## 3. Влияние ключевых признаков на конверсию в покупку страховки

### 3.1 Бренд смартфона и стоимость


In [ ]:
brand_stats = train.groupby('smartphone_brand').agg(
    count=('accepted', 'count'),
    conversion=('accepted', 'mean'),
    avg_price=('smartphone_price', 'mean'),
    avg_income=('monthly_income', 'mean')
).sort_values(by='conversion', ascending=False)
print("Статистика по брендам смартфонов:")
print(brand_stats)


### 3.2 Региональное распределение и образование


In [ ]:
print("Конверсия по регионам:")
print(train.groupby('region')['accepted'].agg(['count', 'mean']).sort_values(by='mean', ascending=False))

print("\nКонверсия по уровню образования:")
print(train.groupby('education')['accepted'].agg(['count', 'mean']).sort_values(by='mean', ascending=False))


## 4. Корреляционный анализ числовых признаков с целевой переменной


In [ ]:
num_cols = train.select_dtypes(include=[np.number]).columns.drop('accepted')
corrs = train[num_cols].apply(lambda c: c.corr(train['accepted'])).sort_values(ascending=False)

print("Корреляция признаков с целевой переменной 'accepted':")
print(corrs)


## 5. Выводы EDA и гипотезы для Feature Engineering
1. **Стоимость смартфона (`smartphone_price`)** — самый сильный предиктор (корреляция 0.71). Клиенты с дорогими телефонами (Apple, Samsung флагманы) значительно чаще покупают страховку экрана.
2. **Доход и баланс (`monthly_income`, `average_monthly_balance`)** — высокая платежеспособность коррелирует с защитой активов.
3. **Отношение стоимости смартфона к доходу (`price_to_income`)** — важный финансовый показатель ценности гаджета для пользователя.
4. **Предыдущий опыт страхования и маркетинговый отклик (`previous_insurance`, `previous_campaign_response`)** — клиенты с опытом страхования охотнее покупают защиту.
5. **Цифровое поведение (`online_payments_share`, `mobile_app_usage`)** — цифровые активные пользователи банка чаще реагируют на диджитал-продукты.
